In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#!pip install -q "numpy<2.0"

In [ ]:
! pip install gradio -q


In [ ]:
import gradio as gr
gr.close_all()

In [ ]:
# !pip install transformers==4.41.2 huggingface_hub==0.23.2 accelerate==0.30.1 tokenizers==0.19.1 -q

In [ ]:
# Force uninstall to clear potential conflicts
#!pip uninstall -y huggingface_hub transformers -q

# Install the latest compatible versions after uninstalling
#!pip install transformers==4.41.2 huggingface_hub==0.23.2 -q

import pickle
# Import transformers after a fresh installation to ensure correct versions are loaded
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [ ]:
# !pip install gradio googletrans==4.0.0-rc1

In [ ]:
# Define the paths where models and tokenizers were saved
load_dir_domain = '/content/drive/MyDrive/Thesis/Models/Cross-Lingual(XLMR)/H_Domain'
load_dir_sub_domain = '/content/drive/MyDrive/Thesis/Models/Cross-Lingual(XLMR)/H_Sub-Domain'
# Load the Domain classification model and tokenizer
loaded_domain_tokenizer = AutoTokenizer.from_pretrained(load_dir_domain)
loaded_domain_model = AutoModelForSequenceClassification.from_pretrained(load_dir_domain)
print(f"Domain model and tokenizer loaded from: {load_dir_domain}")

# Load the Sub-Domain classification model and tokenizer
loaded_sub_domain_tokenizer = AutoTokenizer.from_pretrained(load_dir_sub_domain)
loaded_sub_domain_model = AutoModelForSequenceClassification.from_pretrained(load_dir_sub_domain)
print(f"Sub-Domain model and tokenizer loaded from: {load_dir_sub_domain}")

# Ensure models are on the correct device (CPU/GPU)
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loaded_domain_model.to(device)
loaded_sub_domain_model.to(device)

print("Models are ready for inference.")

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Domain model and tokenizer loaded from: /content/drive/MyDrive/Thesis/Models/Cross-Lingual(XLMR)/H_Domain


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Sub-Domain model and tokenizer loaded from: /content/drive/MyDrive/Thesis/Models/Cross-Lingual(XLMR)/H_Sub-Domain
Models are ready for inference.


In [ ]:
with open('/content/domain_id_to_name.pkl', 'rb') as f:
    domain_label_encoder = pickle.load(f)

with open('/content/subdomain_id_to_name.pkl', 'rb') as f:
    sub_domain_label_encoder = pickle.load(f)

In [ ]:
# import pickle

# # Load the domain_label_encoder.pkl file
# with open('domain_label_encoder.pkl', 'rb') as f:
#     domain_label_encoder = pickle.load(f)

# print("Type of domain_label_encoder:", type(domain_label_encoder))
# print("Available attributes:", dir(domain_label_encoder))

# # Try to safely find where labels are stored
# if hasattr(domain_label_encoder, 'classes_'):
#     print("Labels:", domain_label_encoder.classes_)
# elif hasattr(domain_label_encoder, 'classes'):
#     print("Labels:", domain_label_encoder.classes)
# else:
#     try:
#          # Check if it behaves like a dict or list
#          print("Length/Contents:", len(domain_label_encoder))
#     except Exception as e:
#          print("Could not determine length:", e)

# # Do the same for sub_domain_label_encoder.pkl
# with open('sub_domain_label_encoder.pkl', 'rb') as f:
#     sub_domain_label_encoder = pickle.load(f)

# print("\nType of sub_domain_label_encoder:", type(sub_domain_label_encoder))

In [ ]:
import numpy as np
import pickle
from sklearn.preprocessing import LabelEncoder

# Placeholder for MAX_LENGTH for domain and sub-domain models
# You might need to adjust these based on your model's training configuration
MAX_LENGTH_DOMAIN = 256  # Example value, adjust as per your model
MAX_LENGTH_SUB_DOMAIN = 512 # Example value, adjust as per your model

# # Placeholder for LabelEncoders
# # In a real scenario, these would be loaded from saved files or re-fit on known labels
# # For demonstration, we'll create simple encoders
# # Ensure these labels match the labels your models were trained on.

# # domain_labels = ['Sports' , 'Entertainment','Crime','Agriculture','Business']
# # sub_domain_labels = [
# #     "Violent Crime",
# #     "Cultivation and innovation",
# #     "Cricket",
# #     "Tennis",
# #     "Football",
# #     "TV & Film",
# #     "Economy",
# #     "Music",
# #     "Robbery and Property Crime",
# #     "Markets and products",
# #     "Industry",
# #     "Drugs and Trafficking",
# #     "Policy and administration",
# #     "Global Economy",
# #     "Startups",
# #     "Celebrity"
# # ]



# # The LabelEncoders are already loaded in cell Z4RpZCgiot_g. Removing redundant re-initialization and incorrect fit calls.

print("MAX_LENGTH_DOMAIN and MAX_LENGTH_SUB_DOMAIN variables are set.")

MAX_LENGTH_DOMAIN and MAX_LENGTH_SUB_DOMAIN variables are set.


In [ ]:
def predict_domain(text, model, tokenizer, label_encoder, max_length):
    model.eval() # Set model to evaluation mode
    inputs = tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',
        return_token_type_ids=True,
        truncation=True
    )
    ids = torch.tensor(inputs['input_ids'], dtype=torch.long).unsqueeze(0).to(device)
    mask = torch.tensor(inputs['attention_mask'], dtype=torch.long).unsqueeze(0).to(device)
    token_type_ids = torch.tensor(inputs['token_type_ids'], dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(ids, attention_mask=mask, token_type_ids=token_type_ids)
        logits = outputs.logits

    # Apply softmax to get probabilities and get the predicted class index
    probabilities = torch.softmax(logits, dim=1).cpu().numpy()
    predicted_class_idx = np.argmax(probabilities, axis=1)[0]

    # Decode the predicted class index to the original label using the dictionary
    try:
        predicted_label = label_encoder[predicted_class_idx]
    except KeyError:
        predicted_label = f"Class_{predicted_class_idx}"
        print(f"Warning: Could not find label for index {predicted_class_idx}. Fallback to {predicted_label}")
    return predicted_label

def predict_sub_domain(text, model, tokenizer, label_encoder, max_length):
    model.eval() # Set model to evaluation mode
    inputs = tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',
        return_token_type_ids=True,
        truncation=True
    )
    ids = torch.tensor(inputs['input_ids'], dtype=torch.long).unsqueeze(0).to(device)
    mask = torch.tensor(inputs['attention_mask'], dtype=torch.long).unsqueeze(0).to(device)
    token_type_ids = torch.tensor(inputs['token_type_ids'], dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(ids, attention_mask=mask, token_type_ids=token_type_ids)
        logits = outputs.logits

    # Apply softmax to get probabilities and get the predicted class index
    probabilities = torch.softmax(logits, dim=1).cpu().numpy()
    predicted_class_idx = np.argmax(probabilities, axis=1)[0]

    # Decode the predicted class index to the original label using the dictionary
    try:
        predicted_label = label_encoder[predicted_class_idx]
    except KeyError:
        predicted_label = f"SubClass_{predicted_class_idx}"
        print(f"Warning: Could not find label for index {predicted_class_idx}. Fallback to {predicted_label}")
    return predicted_label

print("Prediction functions 'predict_domain' and 'predict_sub_domain' updated to use dictionary lookup.")

Prediction functions 'predict_domain' and 'predict_sub_domain' updated to use dictionary lookup.


In [ ]:
# Sample text for prediction (you can change this)
# new_sample_text = """ টানা তিনবার বিশ্বকাপে নেই ইতালি। ভাবা যায়! ২০১৪ সালের পর বিশ্বমঞ্চে আজ্জুরিদের নীল জার্সি আর দেখা যায়নি। সর্বশেষ প্লে-অফ ফাইনালে বসনিয়া-হার্জেগোভিনার কাছে হেরে স্বপ্নভঙ্গ হয়েছে চারবারের বিশ্বজয়ীদের। ব্যর্থতার দায় নিয়ে কোচের পদ ছেড়েছেন জেনারো গাত্তুসো। এখন প্রশ্ন একটাই—দিশাহীন এই ইতালিকে পথ দেখাবেন কে?
# # ইতালির সাবেক ডিফেন্ডার লিওনার্দো বোনুচ্চি একটা সমাধান দেখছেন। গাত্তুসো জমানায় তিনি কোচের সহকারী হিসেবে কাজ করেছেন। সেই অভিজ্ঞতা থেকে বোনুচ্চির অনুধাবন, ইতালির ফুটবলে আমূল পরিবর্তনের জন্য এখন ডাগআউটে একজন জাদুকর প্রয়োজন। তাঁর মতে, সেই জাদুকরের নাম পেপ গার্দিওলা!

# # সম্প্রতি স্কাই স্পোর্টসকে দেওয়া এক সাক্ষাৎকারে জুভেন্টাসের এই কিংবদন্তি ডিফেন্ডার বলেছেন, ‘আমরা যদি জাতীয় দলে আমূল পরিবর্তন চাই, তবে পেপ গার্দিওলাই হবেন এই কাজের জন্য যোগ্য লোক। কাজটা কঠিন, তবে স্বপ্ন দেখতে তো পয়সা লাগে না!’

# # ম্যানচেস্টার সিটির সঙ্গে গার্দিওলার চুক্তি আছে ২০২৭ সালের জুন পর্যন্ত। তবে প্রিমিয়ার লিগ ছেড়ে কোনো এক সময় জাতীয় দলের দায়িত্ব নেওয়ার ইচ্ছা গার্দিওলা আগেই প্রকাশ করেছিলেন। সেই সুযোগটাই নিতে চান বোনুচ্চি। অবশ্য সংবাদমাধ্যমে আন্তোনিও কন্তে বা মাসিমিলিয়ানো আল্লেগ্রির নামও আসছে। তবে বোনুচ্চির মন মজেছে পেপ-তত্ত্বে। তাঁর মতে, ইতালিয়ান ফুটবলের নতুন অধ্যায় শুরুর জন্য গার্দিওলাই সেরা কোচ।
# # ২০০৬ সালে বিশ্বজয়ের পর বিশ্বকাপের নকআউট পর্বে আর পা রাখা হয়নি ইতালির। গত দুই আসরের মতো এবারও গ্যালারিতে বসে খেলা দেখতে হবে আজ্জুরি সমর্থকদের। ইতিহাসের প্রথম সাবেক বিশ্বচ্যাম্পিয়ন হিসেবে টানা তিনটি বিশ্বকাপ মিস করার লজ্জার রেকর্ড এখন ইতালির। মাঝখানে ইউরো ২০২০ জয় যেন শুধু একপশলা বৃষ্টির মতো এসেছিল, যা দীর্ঘ খরা কাটাতে পারেনি।
# # বোনুচ্চি মনে করেন, এখন জোড়াতালি দিয়ে চলার সময় শেষ। কারণ, ক্ষত যখন গভীর হয়, তখন ব্যান্ডেজ নয়, অস্ত্রোপচারই শ্রেয়। তাই বোনুচ্চির কথা, ‘বিশ্বের কাছে হারানো সম্মান ফিরে পেতে আমাদের সাহসের সঙ্গে বর্তমান পরিস্থিতির মুখোমুখি হতে হবে। ইতালিয়ান ফুটবলকে একদম শূন্য থেকে আবার গড়তে হবে, শুরুটা হতে হবে রক্ষণভাগ থেকে।’
# # শুধু কোচ বদল করলেই কি সব ঠিক হয়ে যাবে? বোনুচ্চি তেমনটা মনে করেন না। তিনি মনে করেন, রাজনীতি আর ফুটবলের মেলবন্ধন জরুরি। তরুণ প্রতিভাদের বিকশিত হওয়ার সুযোগ দিতে হবে। রাতারাতি কোনো পরিবর্তন সম্ভব নয়। দীর্ঘমেয়াদি পরিকল্পনার গুরুত্ব দিয়ে তিনি বলেছেন, ‘একমাত্র ওষুধ হলো সময়।’ """

new_sample_text= '''
A parked bus in Mymensingh town and another parked at the Daulatdia bus terminal in Rajbari were set on fire. The two buses were torched around Saturday midnight.
The driver and helper went home, parking a bus belonging to Soukhin Paribahan on the road in front of Maskanda Bus Terminal in Mymensingh. The bus was set on fire at around midnight. Local residents subsequently alerted the Fire Service and police.
After analysing CCTV footage from the scene, police said a group of miscreants arriving on two motorcycles had set fire to the bus. Md Pias, 25, was detained from the Maskanda area on suspicion of involvement. He is being questioned.
Hafizur Rahman, station officer of the Bangladesh Agricultural University Fire Service, said they received information about the fire at 12:04 am and two Fire Service units reached the scene within eight minutes. The fire was completely extinguished at 12:40 am.
He added that all the seats and part of the engine were damaged in the fire. The lower section of the bus and its tyres, however, remained intact. Two or three sections of an overhead power line were also damaged by the heat from the fire.
Mohammad Shibirul Islam, officer-in-charge of Kotwali Model police station, told Prothom Alo that CCTV footage from the scene showed four people arriving on two motorcycles. One person had been detained and was being questioned in connection with the incident.
Alamgir Mahmud, president of the Mymensingh Transport Owners’ Association, told Prothom Alo that instructions had been issued to keep buses inside the terminal amid fears that fascists could carry out acts of sabotage. The Soukhin Paribahan bus had been parked on the road and was set on fire. No one was inside the bus when it was torched.
A local bus belonging to Ma Classical Paribahan, parked at the Daulatdia bus terminal in Goalanda upazila of Rajbari, was set on fire at around 1:00 am on Saturday.
Bus workers suspect that the perpetrators used a substance similar to gunpowder to start the fire. Fire Service personnel went to the scene after being alerted and extinguished the blaze.
Workers at the terminal said the Ma Classical Paribahan bus, operated under the Rajbari District Road Transport Owners’ Group, runs on the Daulatdia–Kushtia route. The bus was parked at the Daulatdia terminal after completing its final trip on Saturday night. At around 1:00 am, local residents suddenly saw flames coming from the rear of the bus. They quickly alerted the Goalanda Fire Service Station.
Ujjal Mondal, the driver of the bus, said he parked it at the Daulatdia terminal at around 6:00 pm after completing a trip from Kushtia. He told his assistant to check everything and lock the bus before leaving for his home in Banibohor, Rajbari, at around 6:30 pm. At around 1:30 am, local residents called to inform him that the bus was on fire. He rushed to the terminal and found that the bus had been completely gutted. No one was inside and the doors had been locked.
Quoting witnesses, the driver said local residents suddenly noticed the fire coming from the rear of the bus. The entire bus was engulfed in flames within just five minutes.
“If the fire had started under normal circumstances, the bus could not have burned so quickly,” he said. They suspected that the perpetrators had used a substance similar to gunpowder, causing the entire bus to burn within a very short time.
Mohammad Shafiqul Islam, officer-in-charge of Goalanda Ghat police station, said, “We received the news shortly after 1:00 am. At the time, police officers were patrolling in front of the local market, just a few yards from the bus. They quickly reached the scene and, with the help of local residents, moved the other buses away. Legal action is being taken over the setting of the bus on fire.”
'''

print(f"New Sample Text: {new_sample_text}")
print("________________________")

# 1. Predict Domain using the loaded model
predicted_domain_loaded = predict_domain(
    new_sample_text,
    loaded_domain_model,
    loaded_domain_tokenizer,
    domain_label_encoder,
    MAX_LENGTH_DOMAIN
)
print(f"Predicted Domain (Loaded Model): {predicted_domain_loaded}")
print("________________________")

# 2. Prepare input for Sub-Domain model with the predicted Domain
text_for_sub_domain_loaded = f"{predicted_domain_loaded} {new_sample_text}"

# 3. Predict Sub-Domain using the loaded model
predicted_sub_domain_loaded = predict_sub_domain(
    text_for_sub_domain_loaded,
    loaded_sub_domain_model,
    loaded_sub_domain_tokenizer,
    sub_domain_label_encoder,
    MAX_LENGTH_SUB_DOMAIN
)
print(f"Predicted Sub-Domain (Loaded Model): {predicted_sub_domain_loaded}")

New Sample Text: 
A parked bus in Mymensingh town and another parked at the Daulatdia bus terminal in Rajbari were set on fire. The two buses were torched around Saturday midnight.
The driver and helper went home, parking a bus belonging to Soukhin Paribahan on the road in front of Maskanda Bus Terminal in Mymensingh. The bus was set on fire at around midnight. Local residents subsequently alerted the Fire Service and police.
After analysing CCTV footage from the scene, police said a group of miscreants arriving on two motorcycles had set fire to the bus. Md Pias, 25, was detained from the Maskanda area on suspicion of involvement. He is being questioned.
Hafizur Rahman, station officer of the Bangladesh Agricultural University Fire Service, said they received information about the fire at 12:04 am and two Fire Service units reached the scene within eight minutes. The fire was completely extinguished at 12:40 am.
He added that all the seats and part of the engine were damaged in the fi

In [ ]:
def classify_and_translate(text):
    # Perform domain prediction
    predicted_domain = predict_domain(
        text,
        loaded_domain_model,
        loaded_domain_tokenizer,
        domain_label_encoder,
        MAX_LENGTH_DOMAIN
    )

    # Prepare text for sub-domain prediction by prepending the predicted domain
    text_for_sub_domain = f"{predicted_domain} {text}"

    # Perform sub-domain prediction
    predicted_sub_domain = predict_sub_domain(
        text_for_sub_domain,
        loaded_sub_domain_model,
        loaded_sub_domain_tokenizer,
        sub_domain_label_encoder,
        MAX_LENGTH_SUB_DOMAIN
    )



    return predicted_domain, predicted_sub_domain

In [ ]:
gr.close_all()

interface = gr.Interface(
    fn=classify_and_translate,
    inputs=gr.Textbox(lines=5, placeholder="Enter text here..."),
    outputs=[
        gr.Textbox(label="Predicted Domain"),
        gr.Textbox(label="Predicted Sub-Domain"),

    ],
    title="Domain, Sub-Domain Classification & Translation",
    description="Enter a text to classify its domain and sub-domain, and get its translation (English <-> Bangla)."
)

interface.launch(share=True, debug=True)

Closing server running on port: 7860
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2a067a1d6dabda21be.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
